In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from plot_utils import COST_DIR, load_costs

color_palette = ['#78ACA8', '#E3A39A', '#CA6F6A', '#7B3841', '#D5BC67',
                 '#779674', '#E77A5B', '#9C9DB2', '#20425B', '#E7C4C0', '#235796']
LIMA_COLORS = ['#C0D6CA', '#2D6B8F']  # PatchTST-LiMA, Moment-LiMA (first two rows)
LABELS = {'PatchTST MICA (MLP w/ Query)': 'PatchTST-LiMA', 'Moment MICA (MLP w/ Query)': 'Moment-LiMA',
          'PatchTST LiMA (MLP w/ Query)': 'PatchTST-LiMA', 'Moment LiMA (MLP w/ Query)': 'Moment-LiMA'}

In [ ]:
def sweep(files, xs):
    """(gflops, inference_speed) frames: rows = models, columns = sweep values.
    Sweep points whose CSV is missing are skipped (and reported)."""
    tables, kept = [], []
    for f, x in zip(files, xs):
        if os.path.exists(os.path.join(COST_DIR, f)):
            tables.append(load_costs(f)); kept.append(x)
        else:
            print('missing', f)
    if not tables:
        raise FileNotFoundError(f'no sweep CSVs in {COST_DIR}; see the cell comment for how to make them')
    gflops = pd.concat([t['gflops'] for t in tables], axis=1, keys=kept)
    speed = pd.concat([t['inference_speed'] for t in tables], axis=1, keys=kept)
    gflops.index = speed.index = [LABELS.get(i, i) for i in gflops.index]
    return gflops, speed


def plot_sweep(gflops, speed, xlabel, log=False, ylim=None, save=None):
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['axes.prop_cycle'] = plt.cycler(color=color_palette)
    fontsize = 22
    xs = list(gflops.columns)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    for ax, df, ylabel in [(ax1, gflops, 'GFLOPs'), (ax2, speed, 'Inference Speed')]:
        lines = ax.plot(xs, df.T.values, linewidth=2)
        for line, c in zip(lines, LIMA_COLORS):
            line.set_color(c)
            line.set_linewidth(3.5)
        ax.plot(xs, xs, linestyle='dashed', color='black')
        if log:
            ax.set_yscale('log', base=10)
            ylabel += r' ($\log_{10}$)'
        if ylim:
            ax.set_ylim(ylim)
        ax.set_xlabel(xlabel, fontsize=fontsize)
        ax.set_ylabel(ylabel, fontsize=fontsize)
        ax.tick_params(labelsize=fontsize)
    leg = ax2.legend(list(gflops.index) + ['Linear'], fontsize=fontsize - 4, loc='upper left',
                     frameon=True, bbox_to_anchor=(1.01, 1), borderaxespad=0)
    for i, line in enumerate(leg.get_lines()):
        line.set_linewidth(4 if i < 2 else 2.5)
        if i < 2:
            line.set_color(LIMA_COLORS[i])
    plt.tight_layout()
    if save:
        plt.savefig(save)
    plt.show()

In [ ]:
# Channel-count sweep (L=96). Files come from `MODE=nsweep sbatch run_all_tables.sbatch`.
N_CHANNELS = [7, 15, 20, 50, 100, 150, 200, 300, 400, 500, 600]
gflops, speed = sweep([f'flops_baseline_table_n{n}.csv' for n in N_CHANNELS], N_CHANNELS)
plot_sweep(gflops, speed, 'Channel Count', ylim=[-5, 300])
plot_sweep(gflops, speed, 'Channel Count', log=True, ylim=(0.001, 2500))
#plot_sweep(gflops, speed, 'Channel Count', save='./channel_scale_parameter_impact_figure.pdf')

In [ ]:
# Context-length sweep (C=7). Files come from run_all_tables.sbatch (default mode).
SEQ_LEN = [16, 32, 64, 96, 128, 256, 312, 512, 8192]
gflops, speed = sweep([f'flops_baseline_table_n7_is{L}_MERGED.csv' for L in SEQ_LEN], SEQ_LEN)
plot_sweep(gflops, speed, 'Sequence Length (timesteps)', ylim=[-5, 200])
plot_sweep(gflops, speed, 'Sequence Length (timesteps)', log=True, ylim=(0.001, 2500))
#plot_sweep(gflops, speed, 'Sequence Length (timesteps)', save='./seqlen_scale_parameter_impact_figure.pdf')